<a id="top"></a>

# Lab L0705: wire a tool to a model call

```yaml
title: "Lab L0705: wire a tool to a model call"
keywords: tool calling, tool_calls, ToolMessage, dispatch, round-trip, langchain, bind_tools, lab
estimated duration: 35
```

> **Lesson:** L07. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L07/objectives.md). Solutions: `L0705_lab_solutions.ipynb`. Makes **live** Claude calls — set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running.
>
> **Model-agnostic by design.** The model is a LangChain `ChatAnthropic` (swap for `ChatOpenAI` and the code is unchanged); the tool is a plain typed function bound with `bind_tools`. The tool call arrives as `.tool_calls`; you hand the result back as a `ToolMessage`. The key loads through the config seam (`require_anthropic_key`) — never hard-coded. See the L0703–L0704 demos.
>
> **After this lab you can:** bind a tool to a model, dispatch the returned tool call to a real function, and hand the result back as a `ToolMessage` to get a final answer.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Send the first turn and find the tool call](#2-problem-1--send-the-first-turn-and-find-the-tool-call)
- [3. Problem 2 — Dispatch: run the real function](#3-problem-2--dispatch-run-the-real-function)
- [4. Problem 3 — Continue: hand the result back](#4-problem-3--continue-hand-the-result-back)
- [5. Problem 4 — Why re-send the tool definition? (written)](#5-problem-4--why-re-send-the-tool-definition-written)
- [6. Self-check](#6-self-check)

## 1. Setup

A LangChain `ChatAnthropic` model, the single `calculator` tool (a plain typed function, given), the tool bound with `bind_tools`, and a prompt. Live cells are gated on `LIVE`.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, ToolMessage

from fluffy_potato_curriculum.common.config import get_settings, require_anthropic_key

SONNET = "claude-sonnet-4-6"
LIVE = bool(get_settings().anthropic_api_key)


def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, or raise ValueError on non-arithmetic input.

    Use this whenever the user asks for the result of a calculation.
    """
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    return str(eval(expression))


# The tool is a plain typed function; bind_tools derives the definition (name +
# docstring + argument schema) — no hand-written JSON schema needed.
TOOLS = [calculator]

PROMPT = "What is 6,150 multiplied by 7,012?"

if LIVE:
    model = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=400)
    bound = model.bind_tools(TOOLS)

print("setup ready (LIVE =", LIVE, ")")

[↑ Back to top](#top)

## 2. Problem 1 — Send the first turn and find the tool call

Invoke `bound` with `PROMPT`. From the reply, pull out the first tool call (`reply.tool_calls[0]` — a `{"name", "args", "id"}` dict). Print its `name`, `args`, and `id`.

In [ ]:
if LIVE:
    messages = [HumanMessage(PROMPT)]
    # TODO: first = bound.invoke(messages); then call = first.tool_calls[0]
    #       (a {"name", "args", "id"} dict). Print its name, args, and id.
    raise NotImplementedError("your code here")
else:
    print("offline: set ANTHROPIC_API_KEY to run this lab")

[↑ Back to top](#top)

## 3. Problem 2 — Dispatch: run the real function

Check the tool name is `calculator`, pull the `expression` argument from `call["args"]`, call `calculator(...)`, and capture the result string. This is the application doing the work the model only *proposed*.

In [ ]:
if LIVE:
    # TODO: assert call["name"] == "calculator"; read call["args"]["expression"];
    #       result = calculator(...); print it.
    raise NotImplementedError("your code here")
else:
    print("offline: set ANTHROPIC_API_KEY to run this lab")

[↑ Back to top](#top)

## 4. Problem 3 — Continue: hand the result back

Append the assistant's tool-calling turn (the `AIMessage` itself), then a **`ToolMessage`** carrying your `result` and the same `tool_call_id` as the call. Invoke `bound` on the full conversation and print the model's final text answer (`.text`).

In [ ]:
if LIVE:
    # TODO: messages.append(first)  # the AIMessage tool-calling turn
    #       messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
    #       final = bound.invoke(messages); print(final.text)
    raise NotImplementedError("your code here")
else:
    print("offline: set ANTHROPIC_API_KEY to run this lab")

[↑ Back to top](#top)

## 5. Problem 4 — Why re-send the tool definition? (written)

In Problem 3 you invoked `bound` again on the *second* call, and the bound tool definition rode along with it. In a sentence or two: why is the definition required on every request, not just the first?

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 6. Self-check

Compare against `L0705_lab_solutions.ipynb`. Done when the model proposes a `calculator` call, your dispatch returns `43,123,800`, and the final answer uses that number. You can state why the tool definition rides along on every call (the model is stateless).

[↑ Back to top](#top)